In [1]:
######################################## <<<<  MODEL DOC >>>> ########################################

In [2]:
# import libraries
import numpy as np
import torch
import torch.nn as nn
import sys
from PIL import Image
import matplotlib.pyplot as plt

# import the model image detection
graph_ic = np.load("model/graph.npy", allow_pickle=True)
config_ic = np.load("model/config.npy", allow_pickle=True)
weights_ic = np.load("model/weights.npy", allow_pickle=True)
classes_ic = np.load("model/classes.npy", allow_pickle=True)

In [3]:
### Image Detection Model

In [4]:
# Graph
# indexed by node_id, each entry is a list of input node_ids
# -1 means input from the outside (e.g., input image)
# input node_ids are orderd accroding to the arithmetic order (important for non-commutative operations like concat)

# ------------------------ #
# input node_id: ##0
# ------------------------ #
# output node_id(s): ##142
# ------------------------ #



print("=== Graph ===")
print(f"Type: {type(graph_ic)}, dtype: {graph_ic.dtype}, length: {len(graph_ic)}")
print("First 3 entries:")
for i in range(min(3, len(graph_ic))):
    print(f"graph_ic[{i}] = {graph_ic[i]}")

=== Graph ===
Type: <class 'numpy.ndarray'>, dtype: object, length: 143
First 3 entries:
graph_ic[0] = [-1]
graph_ic[1] = [0]
graph_ic[2] = [1]


In [5]:
# Config

# conv:                         [op_type, in_ch, out_ch, k_h, k_w, s_h, s_w, p_h, p_w, bias]
# activation:                   [op_type, inplace]
# pool:                         [op_type, k_h, k_w, s_h, s_w, p_h, p_w, d_h, d_w, ceil_mode]
# upsample:                     [op_type, scale_h, scale_w, mode]
# concat/resadd:                [op_type] (no params)


print("=== Config ===")
print(f"Type: {type(config_ic)}, dtype: {config_ic.dtype}, length: {len(config_ic)}")
print("First 3 entries:")
for i in range(min(10, len(config_ic))):
    print(f"config_ic[{i}] = {config_ic[i]}")

=== Config ===
Type: <class 'numpy.ndarray'>, dtype: object, length: 143
First 3 entries:
config_ic[0] = ['conv', 3, 16, 6, 6, 2, 2, 2, 2, True]
config_ic[1] = ['activation', True]
config_ic[2] = ['conv', 16, 32, 3, 3, 2, 2, 1, 1, True]
config_ic[3] = ['activation', True]
config_ic[4] = ['conv', 32, 16, 1, 1, 1, 1, 0, 0, True]
config_ic[5] = ['activation', True]
config_ic[6] = ['conv', 16, 16, 1, 1, 1, 1, 0, 0, True]
config_ic[7] = ['activation', True]
config_ic[8] = ['conv', 16, 16, 3, 3, 1, 1, 1, 1, True]
config_ic[9] = ['activation', True]


In [6]:
# Weights

# conv:                         [weight, bias(if exists)]                           <tensor>
# activation:                   []
# pool:                         []
# upsample:                     []
# concat:                       []  
# resadd:                       []



print("=== Weights ===")
print(f"Type: {type(weights_ic)}, dtype: {weights_ic.dtype}, length: {len(weights_ic)}")


## Examples
# print("conv unbiased:", weights_ic[0])
# print("conv biased:", weights_ic[204])
# print("bn:", weights_ic[1])
# print("activation:", weights_ic[2])
# print("pool:", weights_ic[104])
# print("upsample:", weights_ic[112])
# print("concat:", weights_ic[113])
# print("resadd:", weights_ic[18])

=== Weights ===
Type: <class 'numpy.ndarray'>, dtype: object, length: 143


In [7]:
# %% Summary
def inspect_model(graph, config, weights, classes=None, max_nodes=10):
    print(f"{'Node':>4} | {'Op':>12} | {'Inputs':>12} | {'Weights Shapes':>30} | {'Extra Info':>20}")
    print("-"*100)
    for i, (g, c, w) in enumerate(zip(graph, config, weights)):
        if i >= max_nodes:
            break
        op = c[0]
        inputs = str(g)
        if isinstance(w, list) and len(w) > 0:
            shapes = [tuple(t.shape) if isinstance(t, torch.Tensor) else type(t) for t in w]
        else:
            shapes = []
        extra = ""
        if "detect" in op:
            extra = f"Classes={len(classes) if classes is not None else '?'}"
        elif op == "conv":
            _, in_ch, out_ch, k_h, k_w, s_h, s_w, p_h, p_w, bias = c
            extra = f"in={in_ch}, out={out_ch}, k={k_h}x{k_w}, stride={s_h}x{s_w}"
        elif op == "activation":
            extra = f"inplace={c[1]}"
        elif op == "pool":
            extra = f"k={c[1]}x{c[2]}, stride={c[3]}x{c[4]}"
        print(f"{i:>4} | {op:>12} | {inputs:>12} | {str(shapes):>30} | {extra:>20}")
inspect_model(graph_ic, config_ic, weights_ic, classes_ic, max_nodes=2000)

Node |           Op |       Inputs |                 Weights Shapes |           Extra Info
----------------------------------------------------------------------------------------------------
   0 |         conv |         [-1] |         [(16, 3, 6, 6), (16,)] | in=3, out=16, k=6x6, stride=2x2
   1 |   activation |          [0] |                             [] |         inplace=True
   2 |         conv |          [1] |        [(32, 16, 3, 3), (32,)] | in=16, out=32, k=3x3, stride=2x2
   3 |   activation |          [2] |                             [] |         inplace=True
   4 |         conv |          [3] |        [(16, 32, 1, 1), (16,)] | in=32, out=16, k=1x1, stride=1x1
   5 |   activation |          [4] |                             [] |         inplace=True
   6 |         conv |          [5] |        [(16, 16, 1, 1), (16,)] | in=16, out=16, k=1x1, stride=1x1
   7 |   activation |          [6] |                             [] |         inplace=True
   8 |         conv |          [7

In [8]:
######################################## <<<< END OF MODEL DOC >>>> ########################################

In [9]:
######################################## <<<< IMAGE DETECTION RUNTIME >>>> ########################################

In [10]:
######################################## <<<< WITH NO MEMORY OPTIMIZATIONS >>>> ########################################

In [11]:
# Functions
import torch.nn.functional as F
from _utils import cells_to_bboxes, plot_image, non_max_suppression

def execute_layer(x_list, layer_ic):
    print("[exec] num inputs passed:", len(x_list))
    if len(x_list) == 1:   # if more than 1 input, make x a <list> else x a <tensor>
        x = x_list[0]
    else:
        x = x_list
    print("[exec] type of x:", type(x))
    operation = config_ic[layer_ic][0]

    if operation == "conv":
        print(f"{layer_ic}<conv>")
        _, in_ch, out_ch, k_h, k_w, s_h, s_w, p_h, p_w, bias = config_ic[layer_ic]
        w = weights_ic[layer_ic][0]
        b = weights_ic[layer_ic][1] if bias else None
        return F.conv2d(x, w, b, stride=(s_h, s_w), padding=(p_h, p_w))
    elif operation == "activation":
        print(f"{layer_ic}<activation>")
        _, inplace = config_ic[layer_ic]
        return F.silu(x, inplace=inplace)
    elif operation == "pool":
        print(f"{layer_ic}<pool>")
        _, k_h, k_w, s_h, s_w, p_h, p_w, d_h, d_w, ceil_mode = config_ic[layer_ic]
        return F.max_pool2d(
            x,
            kernel_size=(k_h, k_w),
            stride=(s_h, s_w),
            padding=(p_h, p_w),
            dilation=(d_h, d_w),
            ceil_mode=ceil_mode
        )
    elif operation == "upsample":
        print(f"{layer_ic}<upsample>")
        _, scale_h, scale_w, mode = config_ic[layer_ic]
        return F.interpolate(x, scale_factor=(scale_h, scale_w), mode=mode)
    elif operation == "concat":
        print(f"{layer_ic}<concat>")
        return torch.cat(x, dim=1)
    elif operation == "resadd":
        print(f"{layer_ic}<resadd>")
        return x[0] + x[1]
    elif "detect" in operation:
        print(f"{layer_ic}<detect>")
        large  = "detect.l" == operation        # P3/8
        medium = "detect.m" == operation        # P4/16
        small  = "detect.s" == operation        # P5/32
        final  =  "detect.f" == operation       # P3/8 + P4/16 + P5/32

        idx = 0 if large else 1 if medium else 2 if small else -1

        ####### static configs ######    
        nc = len(classes_ic)
        anchors = [
            [(10, 13), (16, 30), (33, 23)],  # P3/8
            [(30, 61), (62, 45), (59, 119)],  # P4/16
            [(116, 90), (156, 198), (373, 326)]  # P5/32#
        ]
        nl = len(anchors)
        no = 5+nc
        na = len(anchors[0])
        stride = [8, 16, 32]
        anchors_ = torch.tensor(anchors).float().view(nl, -1, 2) / torch.tensor(stride).repeat(6, 1).T.reshape(3, 3, 2)
        
        if not final:
            ####### dynamic configs ######
            _, in_ch, out_ch, k_h, k_w, s_h, s_w, p_h, p_w, bias = config_ic[layer_ic]
            ####### dynamic weights ######
            w = weights_ic[layer_ic][0]
            b = weights_ic[layer_ic][1] if bias else None

            ####### pass #######
            # 1. perform convolution
            x = F.conv2d(x, w, b, stride=(s_h, s_w), padding=(p_h, p_w))

            # 2. get shape
            bs, _, ny, nx = x.shape

            # 3. reshape and permute (bs, n_scale_predictions, n_grid_y, n_grid_x, 5 + num_classes)
            return x.view(bs, na, no, ny, nx).permute(0, 1, 3, 4, 2).contiguous()
        else:
            bboxes = cells_to_bboxes(x, anchors_, stride, is_pred=True, to_list=False)
            bboxes = non_max_suppression(bboxes, iou_threshold=0.45, threshold=0.25, tolist=False)
            return bboxes
    else:
        raise ValueError(f"Unsupported operation: {operation}")
    

In [12]:
# Walk the graph

# Memory structure assumptions: for any node_id memory[node_id+1] is the output of the node; prpty, only one output exist
def walker(memory):
    trace = []
    num_nodes = len(graph_ic)
    for node_id in range(num_nodes):
        print("-"*50)
        input_ids = graph_ic[node_id]                               # input node_ids
        inputs = []
        for idx in input_ids:
            print("[walker] input node_id:", idx)
            inputs.append(memory[idx+1])                            # get input from memory
        try:
            trace.append(node_id)
            print("[walker] no of inputs passed:", len(inputs))
            y = execute_layer(inputs, layer_ic=node_id)

            memory[node_id+1] = y # store output in memory  
        except Exception as e:
            import traceback
            print(f"Execution failed at node {node_id}: {e}")
            traceback.print_exc()
            break
    return memory, trace

In [13]:
def preprocess(image_path, new_size=640):
    import cv2
    img = np.array(Image.open(image_path).convert("RGB"))
    h, w = img.shape[:2]
    r = min(new_size / h, new_size / w)
    new_unpad = (int(w * r), int(h * r))
    img_resized = cv2.resize(img, new_unpad)
    canvas = np.full((new_size, new_size, 3), 114, dtype=np.uint8)
    dw = (new_size - new_unpad[0]) // 2
    dh = (new_size - new_unpad[1]) // 2
    canvas[dh:dh+new_unpad[1], dw:dw+new_unpad[0]] = img_resized
    tensor = canvas.transpose((2, 0, 1)).astype(np.float32) / 255.0
    return tensor.tolist(), 3 * new_size * new_size

In [ ]:
import time
im = "/mnt/fileserver/prj/yolo-inference-from-scratch/imgs/000000000872.jpg"

start = time.time()
img, _ = preprocess(im)

img = torch.tensor(img, dtype=torch.float32).unsqueeze(0)

memory = [None for _ in range(1000)] # pre allocation
memory[0] = img                      # store input image in memory at index 0
memory, trace = walker(memory)

plot_image(
    img[0].permute(1, 2, 0).cpu(),
    memory[143],
    classes_ic
)
end = time.time()
print(f"Execution time: {end - start} seconds")

--------------------------------------------------
[walker] input node_id: -1
[walker] no of inputs passed: 1
[exec] num inputs passed: 1
[exec] type of x: <class 'torch.Tensor'>
0<conv>
--------------------------------------------------
[walker] input node_id: 0
[walker] no of inputs passed: 1
[exec] num inputs passed: 1
[exec] type of x: <class 'torch.Tensor'>
1<activation>
--------------------------------------------------
[walker] input node_id: 1
[walker] no of inputs passed: 1
[exec] num inputs passed: 1
[exec] type of x: <class 'torch.Tensor'>
2<conv>
--------------------------------------------------
[walker] input node_id: 2
[walker] no of inputs passed: 1
[exec] num inputs passed: 1
[exec] type of x: <class 'torch.Tensor'>
3<activation>
--------------------------------------------------
[walker] input node_id: 3
[walker] no of inputs passed: 1
[exec] num inputs passed: 1
[exec] type of x: <class 'torch.Tensor'>
4<conv>
--------------------------------------------------
[walke

error: OpenCV(4.13.0) :-1: error: (-5:Bad argument) in function 'rectangle'
> Overload resolution failed:
>  - Layout of the output array img is incompatible with cv::Mat
>  - Layout of the output array img is incompatible with cv::Mat
>  - Expected Ptr<cv::UMat> for argument 'img'
>  - Expected Ptr<cv::UMat> for argument 'img'


In [ ]:
# import yolov5 from torch hub then inference on im
model = torch.hub.load('ultralytics/yolov5', 'yolov5n')
model.eval()
results = model(im)
results.print()
results.show()

In [ ]:
### Debugging and Validation

In [ ]:
### Structure: Conclusion Graph traversal is Higly likely to be Correct ✅


# 1. Our trace
trace_ours = trace

# 2. yolos trace
import torch.fx as fx
trace_yolo = fx.symbolic_trace(model.model.model.model[:24])
# for node in trace_yolo.graph.nodes:
#     print(node.name, node.op, node.target, node.args)


### 3. if this matches most likely the structure is correct
nodes_yolo = list(trace_yolo.graph.nodes)[1:]

for node_id in range(min(len(trace_ours), len(nodes_yolo))):
    ours_op = config_ic[trace_ours[node_id]][0]
    node = nodes_yolo[node_id]
    yolo_op = node.op
    yolo_target = str(node.target)
    match = (
        (ours_op == "conv" and "conv" in yolo_target.lower()) or
        (ours_op == "activation" and yolo_op == "call_module") or
        (ours_op == "resadd" and "add" in yolo_target.lower()) or
        (ours_op == "concat" and "cat" in yolo_target.lower()) or
        (ours_op == "upsample" and "upsample" in yolo_target.lower()) or
        (ours_op == "pool" and "9.m" in yolo_target.lower())
    )
    if not match:
        print("\nMISMATCH FOUND!!\n")
        print(f"Index: {node_id}")
        print(f"Ours : {ours_op}")
        print(f"YOLO : op={yolo_op}, target={yolo_target}")
        continue
else:
    print("\nDone ... \nNote: This test is does not assert correctness/incorrectness.")

In [ ]:
### Weights: Most likely correct ✅
import torch
import torch.fx as fx

weights_yolo = []

traced = fx.symbolic_trace(model.model.model.model[:24])
modules_dict = dict(traced.named_modules())

for node in traced.graph.nodes:
    if node.op == "call_module":
        mod = modules_dict[node.target]
        entry = []
        if hasattr(mod, "weight") and isinstance(mod.weight, torch.Tensor):
            entry.append(mod.weight.detach().clone())
        if hasattr(mod, "bias") and isinstance(mod.bias, torch.Tensor):
            entry.append(mod.bias.detach().clone())
        weights_yolo.append(entry)
    else:
        weights_yolo.append([])
    
weights_yolo = weights_yolo[1:]

for i in range(min(len(weights_yolo), len(weights_ic))):
    w_ic = weights_ic[i]
    w_yolo = weights_yolo[i]
    if len(w_ic) != len(w_yolo):
        print(f"Length mismatch at index {i}: ours={len(w_ic)}, yolo={len(w_yolo)}")
        continue
    for j, (t_ic, t_yolo) in enumerate(zip(w_ic, w_yolo)):
        if t_ic.numel() != t_yolo.numel():
            print(f"Tensor shape mismatch at index {i}, item {j}: ours={t_ic.shape}, yolo={t_yolo.shape}")
            continue
        if not torch.allclose(t_ic, t_yolo, atol=1e-6):
            print(f"Tensor mismatch at index {i}, item {j}: first elements ours={t_ic.view(-1)[0]}, yolo={t_yolo.view(-1)[0]}")
            if j >= 4:  # limit output
                break

In [ ]:
### Config: Most likely correct ✅
trace_ours
trace_yolo

cngf_ours = [config_ic[node_id] for node_id in trace_ours]


import torch

def sniff_yolo_configs(module, prefix=""):
    configs = []
    for name, sub in module.named_children():
        full_name = f"{prefix}.{name}" if prefix else name
        if list(sub.children()):
            configs.extend(sniff_yolo_configs(sub, prefix=full_name))
        else:
            if isinstance(sub, torch.nn.Conv2d):
                kh, kw = sub.kernel_size if isinstance(sub.kernel_size, tuple) else (sub.kernel_size, sub.kernel_size)
                sh, sw = sub.stride if isinstance(sub.stride, tuple) else (sub.stride, sub.stride)
                ph, pw = sub.padding if isinstance(sub.padding, tuple) else (sub.padding, sub.padding)
                cfg = [
                    'conv',
                    sub.in_channels,
                    sub.out_channels,
                    kh, kw,
                    sh, sw,
                    ph, pw,
                    True
                ]
            elif isinstance(sub, torch.nn.SiLU):
                cfg = ['activation', True]
            elif isinstance(sub, torch.nn.BatchNorm2d):
                cfg = ['batchnorm', True]
            elif isinstance(sub, torch.nn.MaxPool2d):
                kh, kw = sub.kernel_size if isinstance(sub.kernel_size, tuple) else (sub.kernel_size, sub.kernel_size)
                sh, sw = sub.stride if isinstance(sub.stride, tuple) else (sub.stride, sub.stride)
                ph, pw = sub.padding if isinstance(sub.padding, tuple) else (sub.padding, sub.padding)
                cfg = ['pool', kh, kw, sh, sw, ph, pw, sub.ceil_mode, False]
            elif isinstance(sub, torch.nn.Upsample):
                scale_h, scale_w = sub.scale_factor if isinstance(sub.scale_factor, tuple) else (sub.scale_factor, sub.scale_factor)
                cfg = ['upsample', scale_h, scale_w, sub.mode]
            else:
                cfg = [sub.__class__.__name__]  # fallback
            configs.append(cfg)
    return configs

yolo_config = sniff_yolo_configs(model.model.model.model)

# a good enough comparison
# make both sets
# remove pools, resadds, concats and add them to two separate sets
# then compare the original sets if they have the same number of elements
# then manualy check the special sets

from collections import Counter

special_layers = {"pool", "resadd", "concat", "Concat", "detect.l", "detect.m", "detect.s", "detect.f"}

yolo_normal = Counter()
yolo_special = Counter()
for layer in yolo_config:
    t = tuple(layer)
    if layer[0] in special_layers:
        yolo_special[t] += 1
    else:
        yolo_normal[t] += 1

ours_normal = Counter()
ours_special = Counter()
for layer in config_ic:
    t = tuple(layer)
    if layer[0] in special_layers:
        ours_special[t] += 1
    else:
        ours_normal[t] += 1

if sum(yolo_normal.values()) - 3 == sum(ours_normal.values()):
    print("Normal layer counts roughly match (YOLO minus 3 convs from detect layer)")
else:
    print("Normal layer count mismatch...")

normal_intersection = yolo_normal & ours_normal
yolo_normal_complement = yolo_normal - normal_intersection
ours_normal_complement = ours_normal - normal_intersection

print("\nNormal layers in YOLO but not in ours:")
for layer, count in yolo_normal_complement.items():
    print(f"{layer} x{count}")

print("\nNormal layers in OURS but not in YOLO:")
for layer, count in ours_normal_complement.items():
    print(f"{layer} x{count}")

yolo_special.update(yolo_normal_complement)
ours_special.update(ours_normal_complement)

special_intersection = yolo_special & ours_special
yolo_special_diff = yolo_special - special_intersection
ours_special_diff = ours_special - special_intersection

print(f"\nLayers in YOLO special set but not in OURS special set ({sum(yolo_special_diff.values())}):")
for layer, count in yolo_special_diff.items():
    print(f"{layer} x{count}")

print(f"\nLayers in OURS special set but not in YOLO special set ({sum(ours_special_diff.values())}):")
for layer, count in ours_special_diff.items():
    print(f"{layer} x{count}")

In [ ]:
map_yolo2us = {
    0: 1,
    1: 3,
    2: 15,
    3: 17,
    4: 34,
    5: 36,
    6: 58,
    7: 60,
    8: 72,
    9: 80,
    10: 82,
    11: 83,
    12: 84,
    13: 95,
    14: 97,
    15: 98,
    16: 99,
    17: 110,
    18: 112,
    19: 113,
    20: 124,
    21: 126,
    22: 127,
    23: 138,
    24: 142
}

In [ ]:
### Hooks for external Layers:

import torch

model = torch.hub.load('ultralytics/yolov5', 'yolov5n', pretrained=True)
model.eval()

memory_yolo = {}



def hook_fn(module, input, output):
    key = f"{module.__class__.__name__}_{len(memory_yolo)}"
    memory_yolo[key] = output.detach() if isinstance(output, torch.Tensor) else (output[0].detach() if isinstance(output, tuple) else output)

# hooks = [m.register_forward_hook(hook_fn) for m in model.modules()]
hooks = [m.register_forward_hook(hook_fn) for m in model.model.model.model]

with torch.no_grad():
    results = model(im)

In [ ]:
### Internal Hooks:
import torch
# model = torch.hub.load('ultralytics/yolov5', 'yolov5n', pretrained=True)
model.eval()

memory_yolo_in = {}

def hook_fn(module, input, output, name):
    if isinstance(output, torch.Tensor):
        memory_yolo_in[name] = output.detach()
    elif isinstance(output, tuple):
        memory_yolo_in[name] = tuple(o.detach() if isinstance(o, torch.Tensor) else o for o in output)
    else:
        memory_yolo_in[name] = output

hooks_in = []

for idx, m in enumerate(model.model.model.model):
    # if m.__class__.__name__.startswith("C3"):
    for name, submodule in m.named_modules():
        if name == '':
            continue
        hook_name = f"{idx}.{name}"
        h = submodule.register_forward_hook(lambda mod, inp, out, n=hook_name: hook_fn(mod, inp, out, n))
        hooks_in.append(h)


with torch.no_grad():
    results = model(im)

print(memory_yolo_in.keys())

In [ ]:
# compare
import torch

def compare_layers(yolo_key, our_node_id, memory_yolo, memory_ours, config_ic, sample_count=5, mode='cr'):
    """
    Compare a YOLO layer with our custom model node.

    Parameters:
    - yolo_key: str, key in memory_yolo
    - our_node_id: int, node index in memory_ours (memory[node_id+1] contains output)
    - memory_yolo: dict of torch.Tensor
    - memory_ours: list of torch.Tensor
    - config_ic: list of layer configs (for names)
    - sample_count: number of elements to compare
    - mode: 'cr' for constrained/interleaved, 'ran' for random sampling
    """
    
    # --- YOLO side ---
    if yolo_key not in memory_yolo:
        print(f"YOLOv5 key '{yolo_key}' not found")
        return
    yolo_feat = memory_yolo[yolo_key]

    # --- Our side ---
    if our_node_id + 1 >= len(memory_ours):
        print(f"Our model node {our_node_id} not found")
        return
    ours_feat = memory_ours[our_node_id + 1]  # memory[node_id+1]

    # Layer names for printing
    yolo_name = yolo_key
    ours_name = config_ic[our_node_id][0] if our_node_id < len(config_ic) else "unknown"

    # Check if tensors and non-empty
    if not isinstance(yolo_feat, torch.Tensor) or not isinstance(ours_feat, torch.Tensor):
        print(f"{yolo_name} || {ours_name} -> non-tensor")
        return
    if yolo_feat.numel() == 0 or ours_feat.numel() == 0:
        print(f"{yolo_name} || {ours_name} -> empty tensor")
        return

    # --- Sampling indices ---
    n = min(yolo_feat.numel(), ours_feat.numel())
    if n < sample_count:
        idx = list(range(n))  # fewer elements than sample_count
    else:
        if mode == 'cr':
            step = n // (sample_count + 1)
            idx = [step * (i + 1) for i in range(sample_count)]
        elif mode == 'ran':
            idx = torch.randint(0, n, (sample_count,)).tolist()
        else:
            raise ValueError("mode must be 'cr' or 'ran'")

    # Flatten tensors
    y_flat = yolo_feat.view(-1)
    o_flat = ours_feat.view(-1)

    y_vals = y_flat[idx].tolist()
    o_vals = o_flat[idx].tolist()

    # Print comparison
    print(f"{yolo_name:<30} || {ours_name:<15}")
    for i in range(len(idx)):
        match_str = 'matches' if (y_vals[i] == o_vals[i]) else 'differs'
        print(f"{match_str:^20}")
        print(f"idx {idx[i]:<6} yolo={y_vals[i]:<12.6f} ours={o_vals[i]:<12.6f}")
    print()

In [ ]:
mapping = {0: '0.conv', 
           1: '0.act', 
           2: '1.conv', 
           3: '1.act', 
           4: '2.cv1.conv',
           5: '2.cv1.act',
           6: '2.m.0.cv1.conv', 
           7: '2.m.0.cv1.act',
           8: '2.m.0.cv2.conv', 
           9: '2.m.0.cv2.act',
           10: '2.m.0',
           11: '2.cv2.conv',
           12: '2.cv2.act',
           13: 'x',
           14: '2.cv3.conv', 
           15: '2.cv3',
           16: '3.conv',
           17: '3.act',}

# '0.conv', '0.act', '1.conv', '1.act', '2.cv1.conv', '2.cv1.act', '2.cv1',
#  '2.m.0.cv1.conv', '2.m.0.cv1.act', '2.m.0.cv1', '2.m.0.cv2.conv', '2.m.0.cv2.act',
#  '2.m.0.cv2', '2.m.0', '2.m', '2.cv2.conv', '2.cv2.act', '2.cv2', '2.cv3.conv',
#  '2.cv3.act', '2.cv3'


In [ ]:
import torch
print("\n=== Start comparison ===")
for ln, yolo_key in enumerate(mapping):
    if yolo_key != 'x':
        print(f"\n=== Comparing Layer {ln} ===")
        compare_layers(mapping[ln], ln, memory_yolo_in, memory, config_ic, sample_count=10, mode='ran')
        print("\n" + "="*110 + "\n")
    else:
        print("this is not comparable layer, skipping...\if suceeding layer passses this works too...")

for h in hooks:
    h.remove()

In [ ]:
# Deep compare
import torch

def compare_deep(yolo_key, our_node_id, memory_yolo, memory_ours):
    if yolo_key not in memory_yolo:
        raise ValueError(f"YOLOv5 key '{yolo_key}' not found")
    yolo_feat = memory_yolo[yolo_key]

    if our_node_id + 1 >= len(memory_ours):
        raise ValueError(f"Our model node {our_node_id} not found")
    ours_feat = memory_ours[our_node_id + 1]

    if not isinstance(yolo_feat, torch.Tensor) or not isinstance(ours_feat, torch.Tensor):
        raise TypeError(f"One of the tensors is not a torch.Tensor: {yolo_key}, node {our_node_id}")

    if yolo_feat.shape != ours_feat.shape:
        raise AssertionError(f"Shape mismatch:\nYOLO: {yolo_feat.shape}\nOurs: {ours_feat.shape}")

    if not torch.equal(yolo_feat, ours_feat):
        diff_count = (yolo_feat != ours_feat).sum().item()
        raise AssertionError(f"Tensors differ! Number of mismatched elements: {diff_count}")

    print(f"[PASS] {yolo_key} vs node {our_node_id}: shapes and values match")

In [ ]:
# A cache to stop redoing time consuming deep comparisons (re run to clear cahce)
comparison_cache = {}

In [ ]:
# Coarse gained deep comparison

# '24.m.0',
# '24.m.1', '24.m.2'

coarse_map = {
    3: '1.act',         # confirms the test itseif working and cnv
    15: '2.cv3.act',    # confirms the test working and c3 for 1-btneck
    34: '4.cv3.act',    # confirms c3 for 2-btneck
    72: '8.cv3.act',    # confirms c3 with all btnecks
    82: '10.act',       # confirms sppf
    95: '13.cv3.act',   # confirms upsample, concat, c3 in neck
    110:'17.cv3.act',
    112: '18.act',
    138: '23.cv3.act',  # confirms upto head

}
for node_id, yolo_key in coarse_map.items():
    if node_id in comparison_cache:
        print(f"Node {node_id} already compared (passed), skipping...")
        continue

    print(f"\n=== Comparing YOLO key '{yolo_key}' with our node {node_id} ===")
    compare_deep(yolo_key, node_id, memory_yolo_in, memory)
    print("\n" + "="*80 + "\n")

    # Add to cache
    comparison_cache[node_id] = True

In [ ]:
## Things should be working iff the HEAD working